# Mol-1 Structure Prediction

[Mol-1](https://github.com/molence/Mol-1) predicts all-atom structures for protein, DNA, RNA, and ligand complexes. This notebook folds the 20-residue Trp-cage mini-protein, then demonstrates a two-chain insulin complex with token-level predicted aligned error (PAE).

See the [Mol-1 technical report](https://github.com/molence/Mol-1/blob/main/assets/report.pdf) for model details. Run the prediction cells on a CUDA GPU; the first call builds the managed environment and downloads the model assets.

In [ ]:
from pathlib import Path

from IPython.display import display

from proto_tools import Chain, Complex
from proto_tools.tools.structure_prediction.mol1 import (
    Mol1Config,
    Mol1Input,
    run_mol1_prediction,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API

In [ ]:
display_api_reference("mol1-prediction", "input", "run_mol1_prediction")

## Config API

In [ ]:
display_api_reference("mol1-prediction", "config", "run_mol1_prediction")

## Output API

In [ ]:
display_api_reference("mol1-prediction", "output", "run_mol1_prediction")

## Basic usage: Trp-cage

Use single-sequence mode to avoid an MSA search and set a seed for repeatable sampling in the same execution environment. All other sampling settings retain their defaults. Mol-1 samples several candidate conformations and returns the highest-ranked structure for each input complex.

In [ ]:
# Trp-cage TC5b is a compact, 20-residue mini-protein.
trpcage_sequence = "NLYIQWLKDGGPSSGRPPPS"
inputs = Mol1Input(complexes=[trpcage_sequence])
config = Mol1Config(use_msa=False, seed=101)

result = run_mol1_prediction(inputs, config)

### Inspect the selected structure

`avg_plddt` uses a 0–100 scale, while `chain_plddt` uses a 0–1 scale. pTM and ipTM also use a 0–1 scale; ipTM is zero for a single chain. Confidence estimates help assess a structural hypothesis and do not establish experimental accuracy.

In [ ]:
trpcage_structure = result.structures[0]
metrics = trpcage_structure.metrics

display({
    "predicted_structures": len(result.structures),
    "protein_length": len(trpcage_sequence),
    "ranking_score": metrics["ranking_score"],
    "avg_plddt (0–100)": metrics["avg_plddt"],
    "chain_plddt (0–1)": metrics["chain_plddt"],
    "ptm (0–1)": metrics["ptm"],
    "gpde (angstroms)": metrics["gpde"],
    "has_clash": metrics["has_clash"],
})

In [ ]:
# Inspect the fold colored by pLDDT confidence.
trpcage_structure.visualize(style="cartoon", color_by="bfactor")

## Advanced usage: a two-chain complex and PAE

Human insulin provides a small protein complex with two chains. This second prediction uses one candidate, fewer denoising steps and recycling cycles, and requests the full PAE matrix. These reduced settings trade sampling and refinement for runtime; they are demonstration settings rather than accuracy recommendations.

PAE measures uncertainty in relative token placement, in angstroms. Its dimensions follow model tokens, which need not correspond one-to-one to residues for ligands or modified residues. Mol-1 also accepts ligand chains specified using the shared `Chain` SMILES or CCD-code fields.

In [ ]:
# Preserve the A and B chain identifiers in the predicted complex.
insulin_complex = Complex(chains=[
    Chain(id="A", sequence="GIVEQCCTSICSLYQLENYCN", entity_type="protein"),
    Chain(id="B", sequence="FVNQHLCGSHLVEALYLVCGERGFFYTPKT", entity_type="protein"),
])
insulin_inputs = Mol1Input(complexes=[insulin_complex])
insulin_config = Mol1Config(
    use_msa=False,
    seed=101,
    num_samples=1,
    num_steps=100,
    num_cycles=10,
    include_pae_matrix=True,
)

insulin_result = run_mol1_prediction(insulin_inputs, insulin_config)

In [ ]:
insulin_structure = insulin_result.structures[0]
insulin_metrics = insulin_structure.metrics
pae = insulin_metrics["pae"]

display({
    "input_chains": len(insulin_complex.chains),
    "avg_plddt (0–100)": insulin_metrics["avg_plddt"],
    "chain_plddt (0–1)": insulin_metrics["chain_plddt"],
    "iptm (0–1)": insulin_metrics["iptm"],
    "avg_pae (angstroms)": insulin_metrics["avg_pae"],
    "pae_shape": (len(pae), len(pae[0])),
})

## Export results

Export the structures as mmCIF using `file_format="cif"`. Each named export directory contains `structure_0.cif`; the structure B-factor values carry pLDDT confidence.

In [ ]:
output_dir = Path("example_output")

result.export(name="trpcage", export_path=output_dir, file_format="cif")
insulin_result.export(name="insulin", export_path=output_dir, file_format="cif")

display(sorted(output_dir.glob("*/structure_*.cif")))